# Gold Pipeline Health Mart

**Purpose**: ETL operational metrics for monitoring pipeline health and data quality

**Target Table**: `workspace.gold.gold_pipeline_health`

**Grain**: One row per pipeline × source × date

**Refresh Strategy**: Full refresh (CREATE OR REPLACE) with metadata logging

**Parameters**:
- `lookback_days`: Number of days of historical data (default: 90)
- `force_full_refresh`: Boolean flag to force table recreation

**Key Metrics**:
- Run counts (total, success, failure, timeout)
- Success rate
- Volume metrics (rows read/written)
- Runtime performance (avg, min, max)
- 7-day rolling averages
- Day-over-day changes
- Anomaly detection flags (volume & performance)

**Data Sources**:
- `workspace.warehouse.fact_pipeline_runs`

**Metadata Logging**:
- Tracks run history in `workspace.metadata.gold_pipeline_health_refresh_log`
- Captures: rows processed, unique pipelines/dates, processing time, status

In [0]:
# Configuration
target_table = "workspace.gold.gold_pipeline_health"
metadata_table = "workspace.metadata.gold_pipeline_health_refresh_log"

# Parameters
lookback_days = 90  # How many days of history to include
force_full_refresh = False  # Set to True to drop and recreate table

In [0]:
import uuid
from datetime import datetime

# Generate unique run ID
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()
status = "RUNNING"
error_message = None

print(f"Run ID: {run_id}")
print(f"Run Timestamp: {run_timestamp}")
print(f"Lookback Days: {lookback_days}")
print(f"Force Full Refresh: {force_full_refresh}")

In [0]:
%sql
-- Create table with all columns if it doesn't exist
CREATE TABLE IF NOT EXISTS workspace.metadata.gold_pipeline_health_refresh_log (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  rows_inserted BIGINT,
  status STRING NOT NULL,
  lookback_days INT,
  rows_processed BIGINT,
  unique_pipelines INT,
  unique_dates INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for gold_pipeline_health refresh operations';

In [0]:
# Optionally drop table for full refresh
if force_full_refresh:
    print(f"Force full refresh enabled - dropping table {target_table}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print("✓ Table dropped")
else:
    print("Proceeding with CREATE OR REPLACE (preserves table properties)")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_pipeline_health (
  health_date_sk INT NOT NULL COMMENT 'Date key',
  pipeline_name STRING NOT NULL COMMENT 'Pipeline name',
  source_name STRING COMMENT 'Source name',
  run_count BIGINT COMMENT 'Total runs',
  success_count BIGINT COMMENT 'Successful runs',
  failure_count BIGINT COMMENT 'Failed runs',
  timeout_count BIGINT COMMENT 'Timed out runs',
  success_rate DECIMAL(10,4) COMMENT 'Success rate',
  total_rows_read BIGINT COMMENT 'Total rows read',
  total_rows_written BIGINT COMMENT 'Total rows written',
  total_runtime_seconds DECIMAL(18,2) COMMENT 'Total runtime',
  avg_runtime_seconds DECIMAL(18,2) COMMENT 'Average runtime',
  max_runtime_seconds DECIMAL(18,2) COMMENT 'Max runtime',
  min_runtime_seconds DECIMAL(18,2) COMMENT 'Min runtime',
  avg_write_ratio DECIMAL(10,4) COMMENT 'Average write ratio',
  avg_rows_per_second DECIMAL(18,2) COMMENT 'Average throughput',
  rolling_7day_avg_rows_written DECIMAL(18,2) COMMENT '7-day rolling avg rows',
  rolling_7day_avg_runtime DECIMAL(18,2) COMMENT '7-day rolling avg runtime',
  pct_change_rows_vs_prev_day DECIMAL(10,4) COMMENT 'Day-over-day row change %',
  pct_change_runtime_vs_prev_day DECIMAL(10,4) COMMENT 'Day-over-day runtime change %',
  volume_anomaly_flag BOOLEAN COMMENT 'Volume anomaly detected',
  performance_anomaly_flag BOOLEAN COMMENT 'Performance anomaly detected',
  created_at TIMESTAMP NOT NULL COMMENT 'Record creation time',
  batch_id STRING NOT NULL COMMENT 'Batch identifier'
)
USING DELTA
COMMENT 'Pipeline operational metrics for monitoring and alerting'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
%sql
INSERT OVERWRITE TABLE workspace.gold.gold_pipeline_health

WITH pipeline_metrics AS (
  SELECT
    fpr.run_date_sk AS health_date_sk,
    fpr.pipeline_name,
    NULL AS source_name,
    fpr.batch_id,
    fpr.run_status AS status,
    
    -- MEASURES: Volume metrics
    fpr.records_read AS rows_read,
    fpr.records_written AS rows_written,
    fpr.duration_minutes * 60 AS runtime_seconds,
    
    -- DERIVED: Data quality metrics
    CASE 
      WHEN fpr.records_read > 0 
      THEN CAST(fpr.records_written AS DECIMAL(10,4)) / fpr.records_read
      ELSE NULL 
    END AS write_ratio,
    
    -- DERIVED: Performance metrics
    CASE 
      WHEN fpr.duration_minutes > 0 
      THEN CAST(fpr.records_written AS DECIMAL(18,2)) / (fpr.duration_minutes * 60)
      ELSE NULL 
    END AS rows_per_second
    
  FROM workspace.warehouse.fact_pipeline_runs fpr
  WHERE fpr.run_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 90), 'yyyyMMdd') AS INT)
),

daily_aggregates AS (
  SELECT
    pm.health_date_sk,
    pm.pipeline_name,
    pm.source_name,
    
    -- MEASURES: Aggregated metrics
    COUNT(*) AS run_count,
    COUNT(CASE WHEN pm.status = 'SUCCESS' THEN 1 END) AS success_count,
    COUNT(CASE WHEN pm.status = 'FAILURE' THEN 1 END) AS failure_count,
    COUNT(CASE WHEN pm.status = 'TIMEOUT' THEN 1 END) AS timeout_count,
    
    SUM(pm.rows_read) AS total_rows_read,
    SUM(pm.rows_written) AS total_rows_written,
    SUM(pm.runtime_seconds) AS total_runtime_seconds,
    
    ROUND(AVG(pm.runtime_seconds), 2) AS avg_runtime_seconds,
    MAX(pm.runtime_seconds) AS max_runtime_seconds,
    MIN(pm.runtime_seconds) AS min_runtime_seconds,
    
    ROUND(AVG(pm.write_ratio), 4) AS avg_write_ratio,
    ROUND(AVG(pm.rows_per_second), 2) AS avg_rows_per_second
    
  FROM pipeline_metrics pm
  GROUP BY pm.health_date_sk, pm.pipeline_name, pm.source_name
),

temporal_enriched AS (
  SELECT
    da.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(da.total_rows_written, 1) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
    ) AS prev_day_rows_written,
    
    LAG(da.avg_runtime_seconds, 1) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
    ) AS prev_day_avg_runtime,
    
    -- 7-day rolling averages
    AVG(da.total_rows_written) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_rows_written,
    
    AVG(da.avg_runtime_seconds) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_runtime
    
  FROM daily_aggregates da
)

SELECT
  -- DIMENSIONS
  te.health_date_sk,
  te.pipeline_name,
  te.source_name,
  
  -- MEASURES: Run counts
  te.run_count,
  te.success_count,
  te.failure_count,
  te.timeout_count,
  
  -- DERIVED: Success rate
  CASE 
    WHEN te.run_count > 0 
    THEN CAST(te.success_count AS DECIMAL(10,4)) / te.run_count
    ELSE NULL 
  END AS success_rate,
  
  -- MEASURES: Volume metrics
  te.total_rows_read,
  te.total_rows_written,
  te.total_runtime_seconds,
  
  -- MEASURES: Performance metrics
  te.avg_runtime_seconds,
  te.max_runtime_seconds,
  te.min_runtime_seconds,
  te.avg_write_ratio,
  te.avg_rows_per_second,
  
  -- TEMPORAL METRICS
  te.rolling_7day_avg_rows_written,
  te.rolling_7day_avg_runtime,
  
  -- DERIVED: Performance changes
  CASE 
    WHEN te.prev_day_rows_written > 0 
    THEN CAST((te.total_rows_written - te.prev_day_rows_written) AS DECIMAL(10,4)) / te.prev_day_rows_written
    ELSE NULL 
  END AS pct_change_rows_vs_prev_day,
  
  CASE 
    WHEN te.prev_day_avg_runtime > 0 
    THEN CAST((te.avg_runtime_seconds - te.prev_day_avg_runtime) AS DECIMAL(10,4)) / te.prev_day_avg_runtime
    ELSE NULL 
  END AS pct_change_runtime_vs_prev_day,
  
  -- DERIVED: Anomaly indicators
  CASE 
    WHEN te.rolling_7day_avg_rows_written > 0 
      AND ABS(te.total_rows_written - te.rolling_7day_avg_rows_written) / te.rolling_7day_avg_rows_written > 0.5
    THEN TRUE
    ELSE FALSE
  END AS volume_anomaly_flag,
  
  CASE 
    WHEN te.rolling_7day_avg_runtime > 0 
      AND te.avg_runtime_seconds > te.rolling_7day_avg_runtime * 1.5
    THEN TRUE
    ELSE FALSE
  END AS performance_anomaly_flag,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
ORDER BY te.health_date_sk DESC, te.pipeline_name;

In [0]:
import time

start_time = time.time()

try:
    # Get summary metrics from the gold table
    metrics = spark.sql(f"""
        SELECT 
            COUNT(*) as rows_processed,
            COUNT(DISTINCT pipeline_name) as unique_pipelines,
            COUNT(DISTINCT health_date_sk) as unique_dates
        FROM {target_table}
    """).collect()[0]
    
    rows_processed = metrics['rows_processed']
    unique_pipelines = metrics['unique_pipelines']
    unique_dates = metrics['unique_dates']
    processing_time = time.time() - start_time
    status = "SUCCESS"
    
    print(f"✓ Processed {rows_processed:,} rows")
    print(f"✓ Unique pipelines: {unique_pipelines}")
    print(f"✓ Unique dates: {unique_dates}")
    print(f"✓ Processing time: {processing_time:.2f}s")
    
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    processing_time = time.time() - start_time
    rows_processed = 0
    unique_pipelines = 0
    unique_dates = 0
    print(f"✗ Error: {error_message}")
    raise

finally:
    # Write metadata log
    spark.sql(f"""
        INSERT INTO {metadata_table} (
            run_id,
            run_timestamp,
            rows_inserted,
            status,
            lookback_days,
            rows_processed,
            unique_pipelines,
            unique_dates,
            force_full_refresh,
            processing_time_seconds,
            error_message
        )
        VALUES (
            '{run_id}',
            TIMESTAMP'{run_timestamp}',
            {rows_processed},
            '{status}',
            {lookback_days},
            {rows_processed},
            {unique_pipelines},
            {unique_dates},
            {force_full_refresh},
            {processing_time:.2f},
            {'NULL' if error_message is None else f"'{error_message}'"}
        )
    """)

In [0]:
%sql
-- Validation: Pipeline health summary
SELECT
  pipeline_name,
  source_name,
  COUNT(*) AS total_days,
  SUM(run_count) AS total_runs,
  ROUND(AVG(success_rate), 4) AS avg_success_rate,
  SUM(total_rows_written) AS total_rows_written,
  ROUND(AVG(avg_runtime_seconds), 2) AS avg_runtime_seconds,
  SUM(CASE WHEN volume_anomaly_flag = TRUE THEN 1 ELSE 0 END) AS volume_anomaly_days,
  SUM(CASE WHEN performance_anomaly_flag = TRUE THEN 1 ELSE 0 END) AS perf_anomaly_days
FROM workspace.gold.gold_pipeline_health
GROUP BY pipeline_name, source_name
ORDER BY total_rows_written DESC;

In [0]:
%sql
-- Overall summary statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT pipeline_name) AS unique_pipelines,
  COUNT(DISTINCT source_name) AS unique_sources,
  COUNT(DISTINCT health_date_sk) AS unique_dates,
  MIN(health_date_sk) AS earliest_date,
  MAX(health_date_sk) AS latest_date,
  SUM(run_count) AS total_runs,
  SUM(success_count) AS total_successes,
  SUM(failure_count) AS total_failures,
  SUM(total_rows_written) AS total_rows_written,
  ROUND(AVG(success_rate), 4) AS avg_success_rate,
  SUM(CASE WHEN volume_anomaly_flag = TRUE THEN 1 ELSE 0 END) AS volume_anomaly_count,
  SUM(CASE WHEN performance_anomaly_flag = TRUE THEN 1 ELSE 0 END) AS perf_anomaly_count
FROM workspace.gold.gold_pipeline_health;

In [0]:
%sql
-- Check for data quality issues
SELECT
  'Null health_date_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_pipeline_health
WHERE health_date_sk IS NULL

UNION ALL

SELECT
  'Null pipeline_name' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_pipeline_health
WHERE pipeline_name IS NULL

UNION ALL

SELECT
  'Negative run counts' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_pipeline_health
WHERE run_count < 0 
   OR success_count < 0 
   OR failure_count < 0

UNION ALL

SELECT
  'Invalid success_rate' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_pipeline_health
WHERE success_rate < 0 OR success_rate > 1

ORDER BY issue_count DESC;

In [0]:
%sql
-- Recent volume or performance anomalies (last 7 days)
SELECT
  health_date_sk,
  pipeline_name,
  source_name,
  total_rows_written,
  rolling_7day_avg_rows_written,
  pct_change_rows_vs_prev_day,
  avg_runtime_seconds,
  rolling_7day_avg_runtime,
  pct_change_runtime_vs_prev_day,
  volume_anomaly_flag,
  performance_anomaly_flag
FROM workspace.gold.gold_pipeline_health
WHERE health_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 7), 'yyyyMMdd') AS INT)
  AND (volume_anomaly_flag = TRUE OR performance_anomaly_flag = TRUE)
ORDER BY health_date_sk DESC, pipeline_name;

In [0]:
%sql
-- View last 10 refresh runs
SELECT 
  run_id,
  run_timestamp,
  lookback_days,
  rows_processed,
  unique_pipelines,
  unique_dates,
  processing_time_seconds,
  status,
  error_message
FROM workspace.metadata.gold_pipeline_health_refresh_log
ORDER BY run_timestamp DESC
LIMIT 10;